# Standards-Based Grading pipeline — runnable demo

This notebook runs the complete SBG starter-kit pipeline on the bundled
synthetic sample data (30 fictitious students): platform exports →
canonical scores → standards achieved → individual student reports.

Based on the MAT188 (University of Toronto) system by Camelia Karimianpour
and collaborators; pipeline adapted from
[dtxe/mat188_learning_standards](https://github.com/dtxe/mat188_learning_standards).


In [1]:
# Get the template (set REPO_URL after you publish it on GitHub).
import os
REPO_URL = 'https://github.com/camelia-k-p/sbg-grading-kit'   # <-- your repo
if os.path.exists('run_pipeline.py'):
    ROOT = '.'                       # running inside the repo already
elif REPO_URL:
    !git clone -q {REPO_URL} sbg-template
    ROOT = 'sbg-template'
else:
    raise SystemExit('Upload the sbg-template folder or set REPO_URL')
%cd {ROOT}
!pip -q install pandas pyyaml openpyxl

/content/sbg-template


## 1. Look at the inputs

`standards.csv` holds the learning standards; the lookup table connects
each standard × modality to the score keys that demonstrate it
(`2|ww1-1,ww1-2,ww1-3` = at least 2 of these 3 correct).


In [2]:
import pandas as pd
display(pd.read_csv('standards/standards.csv'))
display(pd.read_csv('lookup/standards_lookup_table.csv').fillna(''))


,ls_key,unit,type,statement
0,ch0-WRIT-sets,ch0,WRIT,I can describe sets using set builder notation.
1,ch0-COM-vecarith,ch0,COM,I can perform algebraic arithmetic on vectors ...
2,ch0-VG-vecarith,ch0,VG,I can represent vectors arithmetic geometrical...
3,ch0-COM-par,ch0,COM,I can decide whether given vectors are parallel.
4,ch0-COM-per,ch0,COM,I can decide whether given vectors are perpend...
...,...,...,...,...
95,ch8-CON-specthm,ch8,CON,"Given a symmetric matrix A, I can visualize th..."
96,chg-WRIT-matnot,chg,WRIT,I can correctly use mathematical notation.
97,chg-WRIT-matcom,chg,WRIT,I can correctly and effectively communicate id...
98,chg-CON-tf,chg,CON,Given a statement I can decide whether it is t...


,standard,webwork,gateway,tutorial,exam
0,ch0-COM-vecarith,"2|ww1-1,ww1-2,ww1-3",gw1-1,tut1-1-1,
1,ch0-COM-angle,ww1-4,,,
2,ch0-CON-vecline,ww1-5,,tut1-2-1,ex1-1-1
3,ch0-VG-plane,"2|ww1-6,ww1-7,ww1-8",,tut1-3-1,ex1-1-2
4,ch0-WRIT-sets,,,tut2-1-2,ex1-3-1
5,ch1-COM-rref,"2|ww2-1,ww2-2,ww2-3",gw1-2,tut2-1-1,
6,ch1-COM-augset,ww2-5,gw1-3,,
7,ch1-CON-augsoltype,ww2-4,,tut2-2-1,ex1-2-1
8,ch1-CON-matlincom,ww2-6,,,
9,ch1-VG-gslinsys,,,tut2-3-1,ex1-2-2


In [3]:
# A raw WeBWorK export (note the metadata rows — the converter handles them)
print(open('sample_data/webwork/ww1.csv').read()[:600])


NO OF FIELDS,,,,,,,,,,,,
SET NAME,,,,,ww1,ww1,ww1,ww1,ww1,ww1,ww1,ww1
PROB NUMBER,,,,,1,2,3,4,5,6,7,8
CLOSE DATE,,,,,01/01/2030,01/01/2030,01/01/2030,01/01/2030,01/01/2030,01/01/2030,01/01/2030,01/01/2030
CLOSE TIME,,,,,11:00pm EST,11:00pm EST,11:00pm EST,11:00pm EST,11:00pm EST,11:00pm EST,11:00pm EST,11:00pm EST
PROB VALUE,,,,,1,1,1,1,1,1,1,1
STUDENT ID,login ID,LAST NAME,FIRST NAME,SECTION,STATUS,STATUS,STATUS,STATUS,STATUS,STATUS,STATUS,STATUS
7100000001,student01,Lovelace,Ada,,1,1,1,1,1,1,0,0
7100000002,student02,Pascal,Blaise,,1,1,1,1,1,1,1,1
7100000003,student03,Gauss,Carl,,1,1,0,0,0,1,


## 2. Run the pipeline


In [4]:
!python run_pipeline.py config.yml


webwork: sample_data/webwork/gw1.csv
webwork: sample_data/webwork/ww1.csv
webwork: sample_data/webwork/ww2.csv
exam: sample_data/exams/midterm1.csv
tutorials: sample_data/tutorials/tutorial_scores.csv
wrote output/standards_achieved.csv (30, 43)
wrote 30 HTML reports to output/reports_html/


## 3. Inspect the results


In [5]:
sa = pd.read_csv('output/standards_achieved.csv', header=[0,1], index_col=0)
sa['summary'].head(10)


standard,achieved_webwork,tested_webwork,frac_webwork,achieved_gateway,tested_gateway,frac_gateway,achieved_tutorial,tested_tutorial,frac_tutorial,achieved_exam,tested_exam,frac_exam
student01,5,9,0.556,5,6,0.833,3.0,3,1.000,4,5,0.8
student02,8,9,0.889,6,6,1.000,3.0,3,1.000,5,5,1.0
student03,4,9,0.444,4,6,0.667,2.0,5,0.400,2,5,0.4
student04,8,9,0.889,6,6,1.000,3.0,3,1.000,5,5,1.0
student05,2,9,0.222,5,6,0.833,3.0,3,1.000,2,5,0.4
student06,6,9,0.667,3,6,0.500,3.0,5,0.600,5,5,1.0
student07,5,9,0.556,1,6,0.167,1.0,4,0.250,3,5,0.6
student08,8,9,0.889,5,6,0.833,4.0,4,1.000,3,5,0.6
student09,3,9,0.333,3,6,0.500,2.0,3,0.667,4,5,0.8
student10,8,9,0.889,5,6,0.833,5.0,5,1.000,5,5,1.0


In [6]:
# Full detail for one student: 1 = demonstrated, 0 = assessed not yet
# demonstrated, NaN = not yet assessed in that modality
sa.drop(columns='summary', level=0).loc['student01'].unstack('modality')


modality,webwork,gateway,tutorial,exam
standard,,,,
ch0-COM-vecarith,1.0,1.0,NaN,NaN
ch0-COM-angle,1.0,NaN,NaN,NaN
ch0-CON-vecline,1.0,NaN,NaN,1.0
ch0-VG-plane,0.0,NaN,1.0,1.0
ch1-COM-rref,1.0,1.0,NaN,NaN
ch1-COM-augset,1.0,1.0,NaN,NaN
ch1-CON-augsoltype,0.0,NaN,NaN,0.0
ch1-CON-matlincom,0.0,NaN,NaN,NaN
ch2-COM-matvec,0.0,1.0,NaN,NaN


In [7]:
# Render that student's report
from IPython.display import HTML
HTML(open('output/reports_html/student01.html').read())


## 4. Resubmissions: only eventual mastery matters

Append the resubmission export and re-evaluate — demonstrations can only
be added, never lost.


In [8]:
import yaml, sys, os
sys.path.insert(0, 'scripts')
import evaluate_standards
cfg = yaml.safe_load(open('config.yml'))
cfg['tutorial_files'] = ['sample_data/tutorials/tutorial_scores.csv',
                         'sample_data/tutorials/resubmission_scores.csv']
cfg['output_dir'] = 'output_resub'
os.makedirs('output_resub', exist_ok=True)
evaluate_standards.run(cfg)
before = sa['tutorial']
after = pd.read_csv('output_resub/standards_achieved.csv', header=[0,1], index_col=0)['tutorial']
print('new demonstrations:', int(((before != 1) & (after == 1)).sum().sum()))
print('regressions (must be 0):', int(((before == 1) & (after == 0)).sum().sum()))


webwork: sample_data/webwork/gw1.csv
webwork: sample_data/webwork/ww1.csv
webwork: sample_data/webwork/ww2.csv
exam: sample_data/exams/midterm1.csv
tutorials: sample_data/tutorials/tutorial_scores.csv
tutorials: sample_data/tutorials/resubmission_scores.csv
wrote output_resub/standards_achieved.csv (30, 43)
new demonstrations: 34
regressions (must be 0): 0


## 5. Adapt it

1. Edit `standards/standards.csv` with your standards.
2. Label your questions with score keys; fill `lookup/standards_lookup_table.csv`.
3. Point `config.yml` at your real exports.

See `docs/` for the precise data formats and the labeling grammar.
